# C-DM — Fine-tune SD 1.5 theo nhãn multi-hot trên NIH ChestX-ray14

Điều kiện là vector 5 chiều `[No Finding, Infiltration, Effusion, Atelectasis, Others]`,
không dùng prompt. CLIP text encoder bị bỏ khỏi pipeline.

**Trước khi chạy:**
1. Settings → Accelerator → **GPU T4 x2** (script dùng 1 GPU, GPU thứ hai để trống)
2. Settings → Internet → **On**
3. Add Data → tìm `NIH Chest X-rays` (nih-chest-xrays/data) → Add

Thứ tự chạy: `1` → `2` → `3` → `4` (kiểm tra) → `5` (chạy thử 30 step) → `6` (train thật) → `7` (sinh ảnh).


## 1. Lấy code + cài thư viện


In [ ]:
!git clone -q https://github.com/NguyenThanhCong170/C-DM /kaggle/working/C-DM || echo 'đã có sẵn'
%cd /kaggle/working/C-DM
!git pull -q || true
!pip install -q safetensors pyyaml huggingface_hub
!python -c "import torch,torchvision;print('torch',torch.__version__,'| GPU:',torch.cuda.get_device_name(0))"


## 2. Tải base model SD 1.5

Chỉ cần **U-Net + VAE + scheduler** (~1.9 GB). Nhánh multi-hot không dùng
`text_encoder` và `tokenizer` nên bỏ luôn, tiết kiệm ~250 MB và thời gian tải.


In [ ]:
!hf download stable-diffusion-v1-5/stable-diffusion-v1-5 --local-dir /kaggle/working/C-DM/sd15 \
    scheduler/scheduler_config.json \
    vae/config.json vae/diffusion_pytorch_model.fp16.safetensors \
    unet/config.json unet/diffusion_pytorch_model.fp16.safetensors
!du -sh /kaggle/working/C-DM/sd15/*


## 3. Tìm đường dẫn dataset và vá config

Kaggle đôi khi mount dataset ở `/kaggle/input/data`, đôi khi tên khác — cell này tự dò.


In [ ]:
import glob, os, yaml

candidates = glob.glob('/kaggle/input/*/Data_Entry_2017.csv') + glob.glob('/kaggle/input/*/*/Data_Entry_2017.csv')
assert candidates, 'Chưa Add Data "NIH Chest X-rays" vào notebook'
DATA_ROOT = os.path.dirname(candidates[0])
print('DATA_ROOT =', DATA_ROOT)
print('thư mục ảnh:', sorted(os.path.basename(p) for p in glob.glob(DATA_ROOT + '/images_*'))[:14])

# ghi DATA_ROOT vào cả 3 config
for path in ['config/multilabel.yaml', 'config/multilabel_smoke.yaml', 'config/vae_decoder.yaml']:
    cfg = yaml.safe_load(open(path, encoding='utf-8'))
    cfg['data_root'] = DATA_ROOT
    cfg['cache_dir'] = '/kaggle/working/cache512'
    cfg['output_dir'] = '/kaggle/working/' + os.path.basename(path).replace('.yaml','')
    yaml.safe_dump(cfg, open(path,'w',encoding='utf-8'), sort_keys=False, allow_unicode=True)
    print('đã vá', path)


## 4. Kiểm tra toàn bộ setup

Chạy 6 bước: thư viện → base model → dữ liệu → dataset → nạp model → **1 optimizer step thật**.
Bước cuối in VRAM đỉnh và ước tính thời gian của run dài. Nếu cell này pass thì train chạy được.


In [ ]:
!python check_setup.py --config config/multilabel.yaml


## 5. Chạy thử 30 step

Xác nhận loss giảm và ảnh validation ghi ra được, trước khi tốn nhiều giờ.


In [ ]:
!python train_multilabel.py --config config/multilabel_smoke.yaml


In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/kaggle/working/multilabel_smoke/validation/step_*/*.png')):
    print(p); display(Image(p, width=320))


## 6. Train thật

Kaggle giới hạn 12 giờ/session. Nếu `check_setup.py` báo ETA vượt quá, giảm `max_train_steps`
rồi chạy tiếp ở session sau bằng cách thêm vào config:

```yaml
resume_lora: "/kaggle/working/multilabel/lora-10000.safetensors"
resume_label_encoder: "/kaggle/working/multilabel/label_encoder-10000.safetensors"
```

Nhớ **Save Version → Save & Run All** để checkpoint không mất khi hết session.


In [ ]:
!python train_multilabel.py --config config/multilabel.yaml


## 7. Sinh ảnh từ nhãn


In [ ]:
!python generate_multilabel.py \
    --base /kaggle/working/C-DM/sd15 \
    --lora /kaggle/working/multilabel/lora-final.safetensors \
    --label-encoder /kaggle/working/multilabel/label_encoder-final.safetensors \
    --lora-config /kaggle/working/multilabel/lora_config.json \
    --labels 'Effusion|Atelectasis' -n 4 --guidance 4.0 --steps 25 \
    --outdir /kaggle/working/samples


In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/kaggle/working/samples/*.png')):
    print(p); display(Image(p, width=384))


## 8. (Tuỳ chọn) Tinh chỉnh decoder VAE

Stage độc lập — encoder đóng băng nên latent space không đổi, checkpoint LoRA vẫn dùng được.
Chạy xong thì thêm `vae_decoder_checkpoint: <đường dẫn>` vào `config/multilabel.yaml`
hoặc truyền `--vae-decoder` cho `generate_multilabel.py`.


In [ ]:
!python train_vae_decoder.py --config config/vae_decoder.yaml
